# NLP Lab 1: Text Preprocessing Pipeline
**Course**: Introduction to Natural Language Processing  
**Expected Duration**: 2 Hours  

## Objectives
1. Understand the necessity of text preprocessing in NLP workflows.
2. Implement and compare basic preprocessing techniques using Python's built-in string methods.
3. Master library-based tokenization, stopword removal, stemming, and lemmatization using **NLTK** and **spaCy**.
4. Construct a modular preprocessing pipeline and evaluate it on a sample of IMDb movie reviews.

---

### Instructions for Students
* This is the completed reference notebook containing the task descriptions, code implementation, and analysis.
* Run each cell sequentially to observe the behavior, outputs, and comparisons between NLTK and spaCy.

## Environment Setup
First, we need to load the packages we will use during this lab. We'll be using:
* `nltk` (Natural Language Toolkit): A classic library for natural language processing.
* `spacy`: An industry-ready, object-oriented library designed for high-performance NLP tasks.
* `pandas` and `numpy`: Standard data manipulation libraries.

Run the code cell below to download necessary NLTK datasets and initialize spaCy's English pipeline model. If spaCy's model is not installed locally, the cell will attempt to download it for you.

In [ ]:
import nltk
import spacy
import pandas as pd
import numpy as np


print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

print("\nLoading spaCy pipeline...")
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model loaded successfully!")
except OSError:
    print("spaCy model 'en_core_web_sm' not found. Installing it now...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model loaded successfully after download!")

[nltk_data] Downloading package punkt to /Users/jayesh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jayesh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jayesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/jayesh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/jayesh/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/jayesh/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/jayesh/nltk_data...
[nltk_data]   Package averaged_perceptron_ta


Loading spaCy pipeline...
spaCy model loaded successfully!


---
## Phase 1: Tokenization (30 minutes)
Tokenization is the process of breaking a raw sequence of characters (a string of text) into individual, meaningful units called **tokens** (usually words or punctuation marks).

### Task 1.1: Naive Tokenization vs. Real-World Challenges
Before using pre-built libraries, let's see why simple whitespace-splitting is not sufficient. 

**Exercise**: Write a function `naive_tokenize` that uses Python's built-in `.split()` method to tokenize the string `sample_text`. Run it and look closely at the tokens produced.

Naive Tokens:
['I', "don't", 'think', 'this', 'is', 'a', 'good', 'movie.', 'It', 'was', 'too', 'long,', "isn't", 'it?', 'I', 'paid', '$15.50', 'for', 'the', 'ticket.']


### Task 1.2: Tokenization with NLTK
NLTK provides specialized tokenizers that handle contractions and punctuation split rules based on regular expressions and linguistic heuristics. We'll use `word_tokenize` for words and `sent_tokenize` for sentences.

Original Text: I don't think this is a good movie. It was too long, isn't it? I paid $15.50 for the ticket.

--- NLTK Sentences ---
["I don't think this is a good movie.", "It was too long, isn't it?", 'I paid $15.50 for the ticket.']

--- NLTK Words ---
['I', 'do', "n't", 'think', 'this', 'is', 'a', 'good', 'movie', '.', 'It', 'was', 'too', 'long', ',', 'is', "n't", 'it', '?', 'I', 'paid', '$', '15.50', 'for', 'the', 'ticket', '.']


### Task 1.3: Tokenization with spaCy
Unlike NLTK, which treats text as a list of strings, spaCy processes text and returns a rich object-oriented `Doc` structure. The `Doc` contains a sequence of `Token` objects, each carrying its own metadata (such as character offsets, part of speech, punctuation tags, etc.).

**Exercise**: Process the `sample_text` using the loaded spaCy model `nlp`. Iterate through the document to inspect each token, its start index in the original string (`token.idx`), and whether it represents punctuation (`token.is_punct`).

Token Text      | Index    | Is Punctuation?
---------------------------------------------
I               | 0        | False          
do              | 2        | False          
n't             | 4        | False          
think           | 8        | False          
this            | 14       | False          
is              | 19       | False          
a               | 22       | False          
good            | 24       | False          
movie           | 29       | False          
.               | 34       | True           
It              | 36       | False          
was             | 39       | False          
too             | 43       | False          
long            | 47       | False          
,               | 51       | True           
is              | 53       | False          
n't             | 55       | False          
it              | 59       | False          
?               | 61       | True           
I               | 63       | False          
paid     

#### Discussion 1.2:
Compare NLTK's `word_tokenize` output with spaCy's tokenization output. 
1. How did they handle the contraction `"don't"`? Compare their outputs.
2. What about the currency value `"$15.50"`?
3. Why might splitting contractions like `"don't"` into `"do"` and `"n't"` be linguistically useful for downstream tasks (like syntactic parsing or POS tagging)?

**Answers:**
1. **Contraction Handling**: Both NLTK and spaCy split `'don't'` into two separate tokens: `'do'` and `'n't'`. This separates the main verb from the negation particle.
2. **Currency**: Both NLTK and spaCy split `'$15.50'` into the currency symbol `'$'` and the numerical value `'15.50'`. This allows downstream models to easily identify numerical entities and monetary concepts separately.
3. **Linguistic Utility**: Splitting contractions isolates the negation particle (`n't` / `not`) which acts as an adverb modifying the verb (`do`). When doing syntactic parsing or Dependency Grammars, this enables a parser to construct a correct tree where the negation attaches to the verb it modifies.

---
## Phase 2: Stopword Removal (30 minutes)
Stopwords are highly frequent words in a language (like "the", "is", "and", "a") that often carry minimal semantic weight for tasks like text classification or information retrieval. By removing them, we can reduce the dimensionality of our text and focus on the content-carrying words.

### Task 2.1: Stopword Removal with NLTK
NLTK has a built-in corpus of stopwords for several languages. Let's see what is inside and filter our tokens.

Total NLTK stopwords: 198
First 20 stopwords in NLTK list (sorted):
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

Original word count: 27
Filtered word count: 14
Filtered words:
["n't", 'think', 'good', 'movie', '.', 'long', ',', "n't", '?', 'paid', '$', '15.50', 'ticket', '.']


### Task 2.2: Stopword Removal with spaCy
spaCy has a default vocabulary-based stopword checker built directly into its pipeline. We can access it using the `token.is_stop` attribute.

Filtered spaCy tokens:
['think', 'good', 'movie', 'long', 'paid', '$', '15.50', 'ticket']


### Task 2.3: Modifying Stopword Lists for Sentiment Analysis (Critical Thinking)
Standard stopword lists are general-purpose. In certain tasks, removing standard stopwords can destroy vital semantic information. 

Consider the sentence: **"This movie was not good and I never want to see it again."**

If we remove all standard English stopwords, what happens? Let's check.

#### Discussion 2.1:
1. Explain how a sentiment classifier might misinterpret the sentiment of `"This movie was not good"` if the word `"not"` is removed as a stopword.
2. If you were building a sentiment analysis classifier for IMDb reviews, would you use the default stopword list, a customized stopword list, or bypass stopword removal entirely? Justify your choice.

**Answers:**
1. **Sentiment Inversion**: If `'not'` is treated as a stopword and filtered out, the sentence `'This movie was not good'` becomes `'movie good'`. This represents a complete inversion of sentiment from highly negative to positive, which will degrade the performance of sentiment classifiers.
2. **Choice for IMDb Classifier**: I would use a customized stopword list (which explicitly preserves negations like *not*, *no*, *never*, *neither*) or bypass stopword removal entirely. Negations are crucial features in sentiment analysis. Completely removing them creates huge semantic classification errors.

---
## Phase 3: Stemming vs. Lemmatization (40 minutes)
Both stemming and lemmatization aim to reduce inflectional forms (and sometimes derivationally related forms) of a word to a common base form.

*   **Stemming**: A crude, heuristic process that chops off the ends of words. It is fast but often produces non-words (e.g., "university" becomes "univers").
*   **Lemmatization**: A morphologically sound process that uses a vocabulary dictionary and morphological analysis to return the lemma (base dictionary form). It produces real words but is slower and often requires knowing the context (Part of Speech).

### Task 3.1: Stemming with NLTK
NLTK offers multiple stemmers. The two most common are:
1.  **Porter Stemmer**: The oldest and most widely used heuristic stemmer.
2.  **Lancaster Stemmer**: A more aggressive stemmer that often over-stems.

#### Discussion 3.1:
1. Identify cases of **over-stemming** (where two different words are cut down to the same stem, but shouldn't be) in the outputs above.
2. Identify cases of **under-stemming** (where two words that represent the same concept are not reduced to the same stem) in the outputs above.
3. Compare the aggressiveness of `Porter` vs. `Lancaster`. Which one is more aggressive, and how can you tell?

**Answers:**
1. **Over-stemming**: Lancaster stems `'provision'` to `'provid'` (colliding with stem of *provide*). It also stems `'easily'` to `'easy'` and `'fairly'` to `'fair'`, which changes their part of speech. Porter stems `'provision'` to `'provis'`, which is a non-word.
2. **Under-stemming**: Neither stemmer can map the irregular past-tense `'ran'` to `'run'`. Both Porter and Lancaster leave `'ran'` as `'ran'` while `'running'`/`'runs'` become `'run'`. This means different inflectional forms of the same verb remain separate.
3. **Aggressiveness**: The Lancaster stemmer is much more aggressive. For example, `'easily'` becomes `'easy'` (Lancaster) vs. `'easili'` (Porter). `'provision'` becomes `'provid'` (Lancaster) vs. `'provis'` (Porter). Lancaster cuts words down to shorter lengths and is more prone to over-stemming.

### Task 3.2: Lemmatization with NLTK
NLTK uses the WordNet database for lemmatization. Let's see how `WordNetLemmatizer` handles verbs and nouns.

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

print("--- NLTK Lemmatizer without Part of Speech (POS) tags ---")
print(f"{'Word':<15} | {'Lemma':<15}")
print("-" * 33)
for word in ["running", "ran", "runs", "easily", "fairly", "dogs", "leaves"]:
    print(f"{word:<15} | {lemmatizer.lemmatize(word):<15}")

print("\n--- NLTK Lemmatizer with POS tags ---")
print(f"{'Word':<15} | {'POS Tag':<8} | {'Lemma':<15}")
print("-" * 43)
# To lemmatize verbs correctly, we must pass the POS tag pos='v'
# To lemmatize nouns, pos='n' (default)
# To lemmatize adjectives, pos='a'

# Pass the correct POS tag to lemmatizer.lemmatize() for the words:
print(f"{'running':<15} | {'v':<8} | {lemmatizer.lemmatize('running', pos='v'):<15}")
print(f"{'ran':<15} | {'v':<8} | {lemmatizer.lemmatize('ran', pos='v'):<15}")
print(f"{'dogs':<15} | {'n':<8} | {lemmatizer.lemmatize('dogs', pos='n'):<15}")
print(f"{'leaves':<15} | {'v':<8} | {lemmatizer.lemmatize('leaves', pos='v'):<15}")
print(f"{'leaves':<15} | {'n':<8} | {lemmatizer.lemmatize('leaves', pos='n'):<15}")

### Task 3.3: Lemmatization with spaCy
Unlike NLTK, spaCy does not require you to manually pass POS tags. Since spaCy processes the entire sentence as a sequence of words, it automatically tags the Part of Speech in context and uses it to lemmatize accurately.

In [ ]:
sentence = "The leaves are falling as he leaves the house. The dogs were running quickly."
doc_lem = nlp(sentence)

print(f"{'Word':<12} | {'POS Tag (Fine)':<15} | {'POS Tag (Coarse)':<18} | {'Lemma':<12}")
print("-" * 65)

# Loop through tokens in 'doc_lem' and print:
# token.text, token.tag_ (fine-grained POS), token.pos_ (coarse POS), token.lemma_
for token in doc_lem:
    print(f"{token.text:<12} | {token.tag_:<15} | {token.pos_:<18} | {token.lemma_:<12}")

#### Discussion 3.2:
Compare NLTK's Lemmatizer with spaCy's Lemmatizer.
1. Look at the word `"leaves"` in the sentence: `"The leaves are falling as he leaves the house."` How does spaCy distinguish between the noun `"leaves"` and the verb `"leaves"`? What lemmas does it output for each?
2. What are the major pros and cons of using NLTK's `WordNetLemmatizer` compared to spaCy's lemmatizer in a real-world pipeline?

**Answers:**
1. **Disambiguation**: spaCy uses its contextual Part-of-Speech tagger (a neural network trained on large corpus datasets). In the first clause, it tags `'leaves'` as a plural noun (`NNS`/`NOUN`) and resolves it to `'leaf'`. In the second clause, it tags `'leaves'` as a third-person singular verb (`VBZ`/`VERB`) and resolves it to `'leave'`.
2. **Pros & Cons Comparison**:
   * **NLTK Lemmatizer**:
     * *Pros*: Very fast, extremely lightweight (simple dictionary lookup).
     * *Cons*: Requires external POS tagger mapping to function correctly. Without POS tags, it defaults to Nouns and leaves verbs/adjectives unlemmatized.
   * **spaCy Lemmatizer**:
     * *Pros*: Fully integrated pipeline, context-aware lemmatization, handles POS tagging out-of-the-box with state-of-the-art accuracy.
     * *Cons*: Slower execution speed, heavy memory foot-print due to loading deep learning pipeline models.

---
## Phase 4: Advanced Preprocessing Pipeline Challenge (20 minutes)
In a real NLP project, we wrap all our preprocessing steps into a single, optimized function.

### Task 4.1: Building the Pipeline
Write a function `preprocess_text(text, method='nltk')` that:
1.  Converts the text to lowercase.
2.  Tokenizes the text.
3.  Removes stopwords and punctuation, **but keeps negation words** (`"not"`, `"no"`, `"never"`, `"neither"`).
4.  Lemmatizes the tokens:
    *   For `method='nltk'`: First perform part-of-speech tagging (`nltk.pos_tag`) and map the tags to WordNet POS tags (noun, verb, adj, adv) using the helper function `get_wordnet_pos` before calling `WordNetLemmatizer`.
    *   For `method='spacy'`: Process text with `nlp` and extract the lemmas from the tokens.
5.  Joins the cleaned tokens back into a single space-separated string.

In [ ]:
from nltk.corpus import wordnet

def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    # Get the fine-grained POS tag
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def preprocess_text(text, method='nltk'):
    """
    Perform tokenization, custom stopword/punctuation removal, and lemmatization.
    """
    negations = {"not", "no", "never", "neither"}
    
    if method == 'nltk':
        # 1. Lowercase
        text_lower = text.lower()
        
        # 2. Tokenize using nltk
        tokens = word_tokenize(text_lower)
        
        # 3. Filter punctuation and stopwords (keeping negations)
        cleaned_tokens = [w for w in tokens if w.isalnum() and (w not in nltk_stopwords or w in negations)]
        
        # 4. Lemmatize using get_wordnet_pos mapping
        lemmas = [lemmatizer.lemmatize(w, pos=get_wordnet_pos(w)) for w in cleaned_tokens]
        
        return " ".join(lemmas)
        
    elif method == 'spacy':
        # spaCy automatically processes lowercase, POS tag, and lemmatization in its pipeline
        doc_sp = nlp(text.lower())
        
        cleaned_tokens = []
        # Loop through tokens, check if not punctuation and not stopword unless negation
        for token in doc_sp:
            if not token.is_punct and (not token.is_stop or token.text in negations):
                cleaned_tokens.append(token.lemma_)
        
        return " ".join(cleaned_tokens)
    
    else:
        raise ValueError("Method must be 'nltk' or 'spacy'")

# Test your function on a sample sentence
test_sentence = "The actors were not acting well, and the cinematography was disappointing! I will never watch it again."
print("Original :", test_sentence)
print("NLTK Clean:", preprocess_text(test_sentence, method='nltk'))
print("spaCy Clean:", preprocess_text(test_sentence, method='spacy'))

### Task 4.2: Evaluation on IMDb Reviews
Now, let's test your pipeline on a subset of movie reviews. We will also measure the execution time to compare efficiency.

In [ ]:
import time

imdb_reviews = [
    "I absolutely loved this film! The acting was superb and the story kept me hooked from beginning to end.",
    "This was hands down the worst movie of the year. The plot was completely non-existent and the dialog was painful.",
    "It was a decent effort, but fell flat in the second half. Not terrible, but I wouldn't recommend it.",
    "A visual masterpiece. The direction and acting were top-notch, though some parts were a bit slow.",
    "I was extremely disappointed. It was hyped up so much but turned out to be boring and stupid."
]

print("--- NLTK Pipeline Execution ---")
start_time = time.time()
nltk_cleaned = [preprocess_text(review, method='nltk') for review in imdb_reviews]
nltk_time = time.time() - start_time
for orig, clean in zip(imdb_reviews[:3], nltk_cleaned[:3]):
    print(f"Original: {orig}")
    print(f"Cleaned : {clean}\n")
print(f"NLTK processed 5 reviews in {nltk_time:.5f} seconds.")

print("\n--- spaCy Pipeline Execution ---")
start_time = time.time()
spacy_cleaned = [preprocess_text(review, method='spacy') for review in imdb_reviews]
spacy_time = time.time() - start_time
for orig, clean in zip(imdb_reviews[:3], spacy_cleaned[:3]):
    print(f"Original: {orig}")
    print(f"Cleaned : {clean}\n")
print(f"spaCy processed 5 reviews in {spacy_time:.5f} seconds.")

#### Discussion 4.1:
1. Examine the cleaned reviews from NLTK and spaCy. Do you notice any differences in lemmatization quality (e.g., did they handle verbs and nouns like `"loved"`, `"worst"`, `"was"` correctly)?
2. Which library ran faster on these reviews? Why do you think one is faster than the other, and does the speed difference scale when processing millions of reviews?
3. In a production NLP system, how would you balance the trade-off between preprocessing speed and linguistic accuracy?

**Answers:**
1. **Quality differences**: Both pipelines perform well. NLTK lemmatized `'worst'` to `'bad'` (WordNet synonyms resolution) and `'loved'` to `'love'`. spaCy resolved them similarly. However, spaCy is generally more robust with multi-word constructs and contextual ambiguities because it parses sentence structures natively.
2. **Execution Speed**: NLTK was faster. NLTK uses rule-based tokenizers and a direct lexicon lookup table (WordNet) which requires minimal CPU computation. spaCy runs neural network components (transformer-like architectures or CNNs) for dependency parsing and NER, which require much more computational resource. On millions of reviews, this speed gap expands significantly, making optimized NLTK or stripped spaCy pipelines (`nlp.select_pipes(disable=['ner', 'parser'])`) necessary.
3. **Production Trade-off**: I would choose based on downstream models. If training simple bag-of-words or linear models, speed is critical and NLTK suffices. If building advanced transformers or relation extraction systems, grammatical correctness is critical, justifying spaCy's performance cost. Alternatively, spaCy can be accelerated by disabling unused pipeline components.

---
## Conclusion & Submission
Congratulations! You have completed Lab 1.

To submit your lab:
1. Save this notebook (`File > Save and Checkpoint`).
2. Run all cells from the top to ensure your code works without errors (`Cell > Run All`).
3. Export the notebook as requested by your instructor.